# PS-Q1: Biomedical Signal Classifier
## Part 1: ECG Signal Acquisition, Preprocessing & Feature Extraction

### Universal Challenge Framework - Variational Quantum Classifier (VQC) Track
**Objective:** In this notebook, we load and process short electrocardiogram (ECG) segments resembling the **MIT-BIH Arrhythmia Database (PhysioNet)**. We extract 6 clinically established morphological and spectral features, analyze their distributions across Normal and Abnormal cardiac conditions, and normalize them into the quantum rotation angle interval $[0, \pi]$ for angle encoding on a 6-qubit quantum processor.

---
### Clinical Background
Automated screening tools support early detection of cardiac arrhythmias (e.g. Premature Ventricular Contractions - PVC, Ventricular Ectopy).
- **Normal Sinus Rhythm (NSR):** Characterized by periodic P-QRS-T complexes, narrow QRS duration ($<100$ ms), and regular R-R intervals.
- **Abnormal Arrhythmia:** Characterized by ectopic beats, widened and deformed QRS complexes ($>120$ ms), compensatory pauses, elevated beat-to-beat variability, and altered spectral power.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to system path
sys.path.append(os.path.abspath('..'))

from src.feature_extraction import (
    generate_benchmark_dataset,
    extract_ecg_features,
    scale_features_to_quantum_angle
)
from src.evaluation import plot_ecg_segments, plot_feature_distributions

print("Modules imported successfully!")

Modules imported successfully!


### 1. Generating & Inspecting Benchmark ECG Data
We generate 350 calibrated ECG signal segments (sampling rate $f_s = 360$ Hz, duration $T = 2.0$ s), containing normal sinus rhythm and premature ventricular contractions with realistic baseline wander and sensor noise.


In [2]:
# Generate or load dataset
df_features, raw_signals = generate_benchmark_dataset(num_samples=350, abnormal_ratio=0.35, seed=42)

# Display dataset statistics
print(f"Total ECG segments: {len(df_features)}")
print(f"Class Distribution: {df_features['label'].value_counts().to_dict()} (0: Normal, 1: Abnormal)")
df_features.head()

Total ECG segments: 350
Class Distribution: {0: 228, 1: 122} (0: Normal, 1: Abnormal)


,heart_rate_bpm,rr_variability_sdnn,qrs_peak_amplitude,qrs_duration_ms,signal_energy,spectral_power_ratio,label,sample_id
0,68.680445,12.500000,1.148033,50.000000,0.060940,0.646369,0,ECG_0000
1,69.902913,50.000000,1.648684,94.444444,0.154521,0.004498,1,ECG_0001
2,84.872299,195.833333,1.633258,91.666667,0.152128,0.013028,1,ECG_0002
3,71.523179,63.888889,1.729937,91.666667,0.151925,0.005016,1,ECG_0003
4,77.280859,26.388889,1.154999,50.000000,0.062229,0.571844,0,ECG_0004


### 2. Time-Domain ECG Morphology Comparison
Let's inspect the physiological contrast between a normal heartbeat and an abnormal ectopic beat.
Notice the sharp R-peak in Normal Sinus Rhythm versus the widened, inverted QRS complex and altered repolarization in Arrhythmia.


In [3]:
# Plot ECG samples
plot_ecg_segments(raw_signals, df_features['label'].values, fs=360.0)
plt.show()

### 3. Extracted Clinical Features
From each segment, we extract 6 diagnostic metrics:
1. `heart_rate_bpm`: Instantaneous heart rate in beats per minute
2. `rr_variability_sdnn`: Standard deviation of successive R-R intervals (HRV)
3. `qrs_peak_amplitude`: Maximum normalized voltage deflection
4. `qrs_duration_ms`: Estimated duration of ventricular depolarization
5. `signal_energy`: Total waveform power $\frac{1}{N} \sum s(t)^2$
6. `spectral_power_ratio`: Ratio of high-frequency (10-25 Hz) to low-frequency power


In [4]:
feature_cols = [
    "heart_rate_bpm",
    "rr_variability_sdnn",
    "qrs_peak_amplitude",
    "qrs_duration_ms",
    "signal_energy",
    "spectral_power_ratio"
]

plot_feature_distributions(df_features, feature_cols)
plt.show()

C:\Users\CHEMBETI INDU KUMAR\OneDrive\Desktop\fallfest\src\evaluation.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x='Diagnosis', y=col, ax=axes[i], palette=['#3B82F6', '#EF4444'], width=0.45)
C:\Users\CHEMBETI INDU KUMAR\OneDrive\Desktop\fallfest\src\evaluation.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x='Diagnosis', y=col, ax=axes[i], palette=['#3B82F6', '#EF4444'], width=0.45)
C:\Users\CHEMBETI INDU KUMAR\OneDrive\Desktop\fallfest\src\evaluation.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.bo

C:\Users\CHEMBETI INDU KUMAR\OneDrive\Desktop\fallfest\src\evaluation.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x='Diagnosis', y=col, ax=axes[i], palette=['#3B82F6', '#EF4444'], width=0.45)
C:\Users\CHEMBETI INDU KUMAR\OneDrive\Desktop\fallfest\src\evaluation.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x='Diagnosis', y=col, ax=axes[i], palette=['#3B82F6', '#EF4444'], width=0.45)


### 4. Normalizing Features to Quantum Rotation Angles $[0, \pi]$
In quantum circuits, continuous features are encoded via single-qubit rotations:
$$|\psi(\mathbf{x})\rangle = \bigotimes_{i=0}^{5} R_y(\theta_i)|0\rangle, \quad \text{where } \theta_i = \pi \cdot \frac{x_i - \min(x_i)}{\max(x_i) - \min(x_i)} \in [0, \pi]$$

Mapping to $[0, \pi]$ avoids phase ambiguity and periodicity wrap-around ($R_y(0) = I$, $R_y(\pi) = -iY$, flipping $|0\rangle$ to $|1\rangle$).


In [5]:
X_raw = df_features[feature_cols].values
X_quantum, scaler = scale_features_to_quantum_angle(X_raw, feature_range=(0.0, np.pi))

print("Feature value bounds before scaling:")
print(f"  Min: {X_raw.min(axis=0)}")
print(f"  Max: {X_raw.max(axis=0)}")

print("\nFeature value bounds after quantum angle normalization:")
print(f"  Min: {X_quantum.min(axis=0)} rad")
print(f"  Max: {X_quantum.max(axis=0)} rad (<= pi = {np.pi:.4f})")

Feature value bounds before scaling:
  Min: [5.79088472e+01 5.00000000e+00 1.09425834e+00 5.00000000e+01
 4.32479068e-02 3.29638229e-03]
  Max: [171.15689382 250.           2.34290029 105.55555556   0.29341702
   0.86595111]

Feature value bounds after quantum angle normalization:
  Min: [0. 0. 0. 0. 0. 0.] rad
  Max: [3.14159265 3.14159265 3.14159265 3.14159265 3.14159265 3.14159265] rad (<= pi = 3.1416)


### 5. Summary & Next Steps
We have engineered a 6-dimensional feature vector normalized to $[0, \pi]$ ready for angle encoding.
In **Notebook 02**, we build the hybrid Variational Quantum Classifier (VQC), implement class-weighted loss to prioritize **Recall**, and compare against classical Logistic Regression.
